In [0]:
"""
01_fact_operations.py

Manufacturing Operations Fact Table

Business Grain:
    One completed manufacturing operation.

Sources:
    operation_events
    machine_dimension
    operator_dimension

Target:
    fact_operations

Author:
Sumanth Vempalle

Version:
2.3.0
"""

import dlt

from pyspark.sql.functions import col


# ============================================================
# Operations Fact Table
# ============================================================

@dlt.table(
    name="fact_operations",
    comment="Manufacturing Operations Fact Table.",
    table_properties={
        "quality": "gold",
        "pipelines.autoOptimize.managed": "true",
    },
)
def fact_operations():

    operations = dlt.read(
        "operation_events"
    )

    machines = dlt.read(
        "machine_dimension"
    )

    operators = dlt.read(
        "operator_dimension"
    )

    return (

        operations.alias("op")

        .join(

            machines.alias("md"),

            on="machine_id",

            how="left",

        )

        .join(

            operators.alias("od"),

            on="operator_id",

            how="left",

        )

        .select(

            # ====================================================
            # Event
            # ====================================================

            col("op.event_id"),

            col("op.event_timestamp"),

            col("op.event_version"),

            # ====================================================
            # Manufacturing Keys
            # ====================================================

            col("op.plant_code"),

            col("op.hall_id"),

            col("op.line_id"),

            col("op.machine_id"),

            col("op.operator_id"),

            col("op.execution_id"),

            col("op.work_order_id"),

            col("op.product_code"),

            # ====================================================
            # Machine Dimension
            # ====================================================

            col("md.machine_name"),

            col("md.machine_type"),

            col("md.station_code"),

            col("md.station_type"),

            # ====================================================
            # Operator Dimension
            # ====================================================

            col("od.operator_name"),

            col("od.skill_level"),

            # ====================================================
            # Measures
            # ====================================================

            col("op.operation_number"),

            col("op.operation_name"),

            col("op.target_force_kn"),

            col("op.actual_force_kn"),

            col("op.force_deviation_kn"),

            col("op.displacement_mm"),

            col("op.cycle_time_sec"),

            col("op.quality_result"),

            # ====================================================
            # Audit
            # ====================================================

            col("op.silver_processing_timestamp"),

        )

    )